# Natural Language Processing with Disaster Tweets
**Predict which Tweets are about real disasters and which ones are not**

***Acknowledgments***
This dataset was created by the company figure-eight and originally shared on their ‘Data For Everyone’ website here.

Tweet source: https://twitter.com/AnyOtherAnnaK/status/629195955506708480

Kaggle source: https://www.kaggle.com/competitions/nlp-getting-started/overview

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.metrics import accuracy_score, precision_score, f1_score
from sklearn.model_selection import train_test_split

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/festusattornelson/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Step 1. Data Exploration

In [38]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python_Udemy/Projects/NLP-disastertweets/data/disaster_treets_train.csv')

In [39]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 297.5 KB


In [41]:
df.shape

(7613, 5)

In [42]:
df['keyword'] = df['keyword'].fillna('none')
df['location'] = df['location'].fillna('Unknown')

print("mising values after cleaning")
df.isnull().sum()

mising values after cleaning


id          0
keyword     0
location    0
text        0
target      0
dtype: int64

In [43]:
df['char_count'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().apply(len)
df['hashtag_count'] = df['text'].str.count(r'#\w+')
df['mention_count'] = df['text'].str.count(r'@\w+')
df['url_count'] = df['text'].str.count(r'http\S+')

df[['text','char_count','word_count','hashtag_count','mention_count','url_count']].head()

,text,char_count,word_count,hashtag_count,mention_count,url_count
0,Our Deeds are the Reason of this #earthquake M...,69,13,1,0,0
1,Forest fire near La Ronge Sask. Canada,38,7,0,0,0
2,All residents asked to 'shelter in place' are ...,133,22,0,0,0
3,"13,000 people receive #wildfires evacuation or...",65,8,1,0,0
4,Just got sent this photo from Ruby #Alaska as ...,88,16,2,0,0


In [44]:
df_train =pd.DataFrame({'text':df['text'], 'target':df['target']})

In [45]:
df_train.head()

,text,target
0,Our Deeds are the Reason of this #earthquake M...,1
1,Forest fire near La Ronge Sask. Canada,1
2,All residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,Just got sent this photo from Ruby #Alaska as ...,1


In [46]:
df_train.target.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

## Step 2. Preprocessing

In [47]:
# Lowercasing
df_train['text'] = df_train['text'].apply(lambda text: text.lower())
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,"13,000 people receive #wildfires evacuation or...",1
4,just got sent this photo from ruby #alaska as ...,1


In [48]:
# Cleaning - remove numbers
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'\d+', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this #earthquake m...,1
1,forest fire near la ronge sask. canada,1
2,all residents asked to 'shelter in place' are ...,1
3,", people receive #wildfires evacuation orders ...",1
4,just got sent this photo from ruby #alaska as ...,1


In [49]:
# Removing hashtags and special characters
df_train['text'] = df_train['text'].apply(lambda text: re.sub(r'[^a-zA-Z\s]', '', text))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [50]:
# Removing punctuation
df_train['text'] = df_train['text'].apply(lambda text: text.translate(str.maketrans('', '', string.punctuation)))
df_train.head()

,text,target
0,our deeds are the reason of this earthquake ma...,1
1,forest fire near la ronge sask canada,1
2,all residents asked to shelter in place are be...,1
3,people receive wildfires evacuation orders in...,1
4,just got sent this photo from ruby alaska as s...,1


In [51]:
# Tokenization
df_train['text'] = df_train['text'].apply(lambda text: text.split())
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [52]:
# Filtering out non-alphabetic characters and short tokens
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word.isalpha() and len(word) > 1   ])
df_train.head()

,text,target
0,"[our, deeds, are, the, reason, of, this, earth...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[all, residents, asked, to, shelter, in, place...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[just, got, sent, this, photo, from, ruby, ala...",1


In [53]:
# Stop words removal
stop_words = set(stopwords.words('english'))
df_train['text'] = df_train['text'].apply(lambda text: [word for word in text if word not in stop_words])
df_train.head()

,text,target
0,"[deeds, reason, earthquake, may, allah, forgiv...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[residents, asked, shelter, place, notified, o...",1
3,"[people, receive, wildfires, evacuation, order...",1
4,"[got, sent, photo, ruby, alaska, smoke, wildfi...",1


In [54]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
df_train['text'] = df_train['text'].apply(lambda text: [lemmatizer.lemmatize(word) for word in text])
df_train.head()

,text,target
0,"[deed, reason, earthquake, may, allah, forgive...",1
1,"[forest, fire, near, la, ronge, sask, canada]",1
2,"[resident, asked, shelter, place, notified, of...",1
3,"[people, receive, wildfire, evacuation, order,...",1
4,"[got, sent, photo, ruby, alaska, smoke, wildfi...",1


In [55]:
df_train['text'] = df_train['text'].apply(lambda x: ' '.join(x))

## Step 3. Feature Extraction

### 3.1 Data Split

In [56]:
# import trian-test library
from sklearn.model_selection import train_test_split

In [57]:
X = df['text']
y = df['target']

In [58]:
# Split into training and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, random_state=42, stratify=y)

In [59]:
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 6090
Test set size: 1523


### 3.2 Count and TF-IDF Vectorizers

In [60]:
# import vectorizer libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [61]:
# Defining the vectorizers
vectorizers = {
    "CountVectorizer": CountVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    ),
    "TF-IDF": TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2),
        stop_words='english'
    )
}

In [63]:
for vec_name, vectorizer in vectorizers.items():
    print(f"Training {vec_name} completed")
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

Training CountVectorizer completed
Training TF-IDF completed


## Step 4. Model Training and Evaluation

In [64]:
# model libraries
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

In [65]:
# Defining models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Linear SVC": LinearSVC()
}

In [66]:
results = []

for model_name, model in models.items():
    print(f"Training {model_name} successful")

    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Vectorizer": vec_name,
        "Model": model_name,
        "Accuracy": accuracy,
        "F1_Score": f1
    })

Training Logistic Regression successful
Training Decision Tree successful
Training Random Forest successful
Training Linear SVC successful


In [67]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

In [68]:
# Sorting the F1-Score
results_df = results_df.sort_values(
    by="F1_Score",
    ascending=False
)

In [69]:
print(results_df)

  Vectorizer                Model  Accuracy  F1_Score
0     TF-IDF  Logistic Regression  0.820092  0.772803
3     TF-IDF           Linear SVC  0.791202  0.748815
2     TF-IDF        Random Forest  0.783979  0.725146
1     TF-IDF        Decision Tree  0.731451  0.683681


In [70]:
# Selecting the Baseline model
best_baseline = results_df.iloc[0]

## Step 5. Hyperparameter Tuning

### 5.1 GridSearch Hayperparameter Tuning

In [71]:
# Defining the Pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ("Vectorizer", TfidfVectorizer()),
    ("model", LogisticRegression())
])

## Step 6. Check incorrect predictions